In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import pickle
import json
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import warnings
warnings.filterwarnings('ignore')


In [22]:
# Load the dataset
file_path = r"C:\Users\Priyanshi\sih\backend\datasets\Crop_recommendation.csv"
df = pd.read_csv(file_path)

print(f"Dataset shape: {df.shape}")
print(f"Unique crops: {df['label'].nunique()}")
print("Dataset loaded successfully!")


Dataset shape: (2200, 8)
Unique crops: 22
Dataset loaded successfully!


In [23]:
# Add minimal noise to prevent overfitting
def add_minimal_noise(df, noise_level=0.05):
    """Add 5% noise to features to prevent overfitting"""
    
    features = df.drop('label', axis=1)
    labels = df['label']
    
    # Add small amount of noise to features
    noisy_features = features.copy()
    for col in features.columns:
        feature_std = features[col].std()
        noise = np.random.normal(0, feature_std * noise_level, len(features))
        noisy_features[col] = features[col] + noise
        
        # Keep within bounds
        min_val = features[col].min()
        max_val = features[col].max()
        noisy_features[col] = np.clip(noisy_features[col], min_val, max_val)
    
    # Combine and shuffle
    result_df = pd.concat([noisy_features, labels], axis=1)
    result_df = shuffle(result_df, random_state=42).reset_index(drop=True)
    
    print(f"Added {noise_level*100}% noise to prevent overfitting")
    return result_df

# Apply minimal noise
df_final = add_minimal_noise(df, noise_level=0.05)
print(f"Final dataset shape: {df_final.shape}")


Added 5.0% noise to prevent overfitting
Final dataset shape: (2200, 8)


In [24]:
# Prepare data for training
X = df_final.drop('label', axis=1)
y_text = df_final['label']

# Encode labels
encoder = LabelEncoder()
y = encoder.fit_transform(y_text)

print(f"Features: {list(X.columns)}")
print(f"Number of classes: {len(encoder.classes_)}")
print(f"Classes: {list(encoder.classes_)}")


Features: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
Number of classes: 22
Classes: ['apple', 'banana', 'blackgram', 'chickpea', 'coconut', 'coffee', 'cotton', 'grapes', 'jute', 'kidneybeans', 'lentil', 'maize', 'mango', 'mothbeans', 'mungbean', 'muskmelon', 'orange', 'papaya', 'pigeonpeas', 'pomegranate', 'rice', 'watermelon']


In [25]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"All classes present in training: {len(np.unique(y_train)) == len(np.unique(y))}")


Training set size: 1760
Test set size: 440
All classes present in training: True


In [26]:
# Train XGBoost model
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='mlogloss'
)

print("Training XGBoost model...")
model.fit(X_train, y_train)

# Evaluate model
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_accuracy = accuracy_score(y_train, train_pred)
test_accuracy = accuracy_score(y_test, test_pred)

print(f"\nTraining Accuracy: {train_accuracy*100:.2f}%")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Overfitting Gap: {(train_accuracy - test_accuracy)*100:.2f}%")

if 85 <= test_accuracy*100 <= 95:
    print("✅ Model accuracy is in realistic range!")
else:
    print("⚠️ May need parameter adjustment")


Training XGBoost model...

Training Accuracy: 100.00%
Test Accuracy: 99.55%
Overfitting Gap: 0.45%
⚠️ May need parameter adjustment


In [27]:
# K-Fold Cross Validation to verify no overfitting
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Performing 5-fold cross-validation to check for overfitting...")

# Create a copy of the model for CV
cv_model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='mlogloss'
)

# 5-fold stratified cross validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(cv_model, X_train, y_train, cv=kfold, scoring='accuracy')

print(f"\nCross-Validation Results:")
print(f"CV Scores: {[f'{score:.3f}' for score in cv_scores]}")
print(f"Mean CV Accuracy: {cv_scores.mean():.3f} ({cv_scores.mean()*100:.1f}%)")
print(f"CV Standard Deviation: {cv_scores.std():.3f} ({cv_scores.std()*100:.1f}%)")
print(f"CV Range: {cv_scores.min():.3f} - {cv_scores.max():.3f}")

# Compare with test accuracy
cv_gap = abs(test_accuracy - cv_scores.mean())
print(f"\nOverfitting Check:")
print(f"Test Accuracy: {test_accuracy:.3f} ({test_accuracy*100:.1f}%)")
print(f"CV Mean Accuracy: {cv_scores.mean():.3f} ({cv_scores.mean()*100:.1f}%)")
print(f"Gap between Test and CV: {cv_gap:.3f} ({cv_gap*100:.1f}%)")

if cv_gap < 0.03:  # Less than 3% difference
    print("✅ EXCELLENT: No overfitting detected!")
elif cv_gap < 0.05:  # Less than 5% difference
    print("✅ GOOD: Minimal overfitting")
else:
    print("⚠️ WARNING: Possible overfitting detected")

if cv_scores.std() < 0.02:  # Low variance between folds
    print("✅ STABLE: Model is consistent across folds")
else:
    print("⚠️ Model shows variance across folds")


Performing 5-fold cross-validation to check for overfitting...

Cross-Validation Results:
CV Scores: ['0.986', '0.983', '0.977', '0.983', '0.986']
Mean CV Accuracy: 0.983 (98.3%)
CV Standard Deviation: 0.003 (0.3%)
CV Range: 0.977 - 0.986

Overfitting Check:
Test Accuracy: 0.995 (99.5%)
CV Mean Accuracy: 0.983 (98.3%)
Gap between Test and CV: 0.012 (1.2%)
✅ EXCELLENT: No overfitting detected!
✅ STABLE: Model is consistent across folds


In [28]:
# Save the model and class mapping (only if not already exists)
import os

print("Checking if files need to be saved...")

# Define file paths
mapping_path = '../crop_class_mapping_final.json'
model_path = '../crop_recommendation_model_final.pkl'
encoder_path = '../label_encoder_final.pkl'

# Create class mapping
class_mapping = {i: label for i, label in enumerate(encoder.classes_)}

# Check and save class mapping
if not os.path.exists(mapping_path):
    with open(mapping_path, 'w') as f:
        json.dump(class_mapping, f, indent=4)
    print(f"✅ Class mapping saved to: {mapping_path}")
else:
    print(f"⏭️ Class mapping already exists: {mapping_path}")

# Check and save model
if not os.path.exists(model_path):
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"✅ Model saved to: {model_path}")
else:
    print(f"⏭️ Model already exists: {model_path}")

# Check and save encoder
if not os.path.exists(encoder_path):
    with open(encoder_path, 'wb') as f:
        pickle.dump(encoder, f)
    print(f"Label encoder saved to: {encoder_path}")
else:
    print(f"⏭ Label encoder already exists: {encoder_path}")

print("\n File saving complete!")


Checking if files need to be saved...
⏭️ Class mapping already exists: ../crop_class_mapping_final.json
⏭️ Model already exists: ../crop_recommendation_model_final.pkl
⏭ Label encoder already exists: ../label_encoder_final.pkl

 File saving complete!


In [29]:
# Quick test with sample prediction
def test_prediction(N, P, K, temperature, humidity, ph, rainfall, description="Test"):
    """Test the model with sample conditions"""
    
    input_data = pd.DataFrame({
        'N': [N], 'P': [P], 'K': [K],
        'temperature': [temperature], 'humidity': [humidity],
        'ph': [ph], 'rainfall': [rainfall]
    })
    
    pred_idx = model.predict(input_data)[0]
    pred_proba = model.predict_proba(input_data)[0]
    
    predicted_crop = encoder.inverse_transform([pred_idx])[0]
    confidence = pred_proba.max() * 100
    
    # Get top 3 recommendations
    top_3_indices = pred_proba.argsort()[-3:][::-1]
    top_3_crops = [encoder.inverse_transform([idx])[0] for idx in top_3_indices]
    top_3_probs = [pred_proba[idx] * 100 for idx in top_3_indices]
    
    print(f"\n🌾 {description}")
    print(f"Conditions: N={N}, P={P}, K={K}, Temp={temperature}°C, Humidity={humidity}%, pH={ph}, Rain={rainfall}mm")
    print(f"🎯 RECOMMENDED: {predicted_crop.upper()} ({confidence:.1f}%)")
    print(f"Top 3: {top_3_crops[0]} ({top_3_probs[0]:.1f}%), {top_3_crops[1]} ({top_3_probs[1]:.1f}%), {top_3_crops[2]} ({top_3_probs[2]:.1f}%)")
    
    return predicted_crop, confidence

# Test with realistic scenarios
print("Testing model with sample conditions:")

test_prediction(85, 45, 40, 25, 80, 6.5, 200, "Rice Growing Conditions")
test_prediction(70, 35, 55, 32, 45, 7.2, 85, "Cotton Growing Conditions") 
test_prediction(75, 50, 42, 18, 70, 6.5, 150, "Apple Growing Conditions")
test_prediction(60, 40, 35, 28, 85, 6.2, 280, "Tropical Conditions")

print("\n🎉 Model is ready for deployment!")


Testing model with sample conditions:

🌾 Rice Growing Conditions
Conditions: N=85, P=45, K=40, Temp=25°C, Humidity=80%, pH=6.5, Rain=200mm
🎯 RECOMMENDED: JUTE (51.8%)
Top 3: jute (51.8%), rice (47.7%), coffee (0.0%)

🌾 Cotton Growing Conditions
Conditions: N=70, P=35, K=55, Temp=32°C, Humidity=45%, pH=7.2, Rain=85mm
🎯 RECOMMENDED: MANGO (88.2%)
Top 3: mango (88.2%), maize (3.3%), papaya (2.0%)

🌾 Apple Growing Conditions
Conditions: N=75, P=50, K=42, Temp=18°C, Humidity=70%, pH=6.5, Rain=150mm
🎯 RECOMMENDED: JUTE (87.2%)
Top 3: jute (87.2%), maize (3.6%), coffee (1.1%)

🌾 Tropical Conditions
Conditions: N=60, P=40, K=35, Temp=28°C, Humidity=85%, pH=6.2, Rain=280mm
🎯 RECOMMENDED: RICE (98.2%)
Top 3: rice (98.2%), jute (0.6%), mungbean (0.1%)

🎉 Model is ready for deployment!
